In [1]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.datasets import wage_panel
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

from pymargins import Margins, drop_outliers, reimpute, trim

cols = ["lwage", "exper", "educ", "married", "union"]
df = wage_panel.load().reset_index(drop=True)[cols].copy()
print(df.describe().round(2))

         lwage    exper     educ  married    union
count  4360.00  4360.00  4360.00  4360.00  4360.00
mean      1.65     6.51    11.77     0.44     0.24
std       0.53     2.83     1.75     0.50     0.43
min      -3.58     0.00     3.00     0.00     0.00
25%       1.35     4.00    11.00     0.00     0.00
50%       1.67     6.00    12.00     0.00     0.00
75%       1.99     9.00    12.00     1.00     0.00
max       4.05    18.00    16.00     1.00     1.00


In [2]:
rng = np.random.default_rng(7)
p_miss = 1 / (1 + np.exp(-(df["exper"] - df["exper"].mean()) / 2))
miss = rng.uniform(size=len(df)) < 0.30 * p_miss

df_nan = df.copy()
df_nan.loc[miss, "educ"] = np.nan
print(f"missing educ: {int(miss.sum())} rows ({miss.mean():.1%})")

missing educ: 654 rows (15.0%)


In [3]:
df_init = df_nan.fillna(df_nan.mean(numeric_only=True))
fit = smf.ols("lwage ~ exper + educ + married + union", data=df_init).fit()
print(fit.params.round(4))

Intercept    0.1010
exper        0.0429
educ         0.0986
married      0.1414
union        0.1730
dtype: float64


In [4]:
imp = IterativeImputer(max_iter=10, random_state=0, sample_posterior=True)


def imputer(frame):
    return pd.DataFrame(imp.fit_transform(frame), columns=frame.columns)

In [5]:
m_naive = Margins.linear_scale(fit, method="bootstrap", n_boot=500, rng_seed=3)
print(m_naive.dydx("educ").summary())

           Margins Result (bootstrap, level=0.95)          
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1009   0.0050     0.1009  0.000    0.0912, 0.1100

n = 4360
κ: 0.000


In [6]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # silence the imputer's convergence chatter
    m_mi = Margins.linear_scale(
        fit,
        transforms=[reimpute(imputer, incomplete=df_nan)],
        method="bootstrap",
        n_boot=500,
        rng_seed=3,
    )
print(m_mi.dydx("educ").summary())

           Margins Result (bootstrap, level=0.95)          
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1009   0.0054     0.1009  0.000    0.0943, 0.1152

n = 4360
κ: 0.000


In [7]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_compose = Margins.linear_scale(
        fit,
        transforms=[
            reimpute(imputer, incomplete=df_nan),
            trim(lower=2.0, columns=["educ"]),
        ],
        method="bootstrap",
        n_boot=400,
        rng_seed=3,
    )
print(m_compose.dydx("educ").summary())

           Margins Result (bootstrap, level=0.95)          
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1009   0.0055     0.1009  0.000    0.0942, 0.1139

n = 4360
κ: 0.000


In [8]:
def far_below(frame):
    med = frame["lwage"].median()
    mad = (frame["lwage"] - med).abs().median()
    return frame["lwage"] < med - 5 * mad


print(f"flagged on the full sample: {int(far_below(df).sum())} rows")

df_clean = df[~far_below(df)].reset_index(drop=True)
fit_clean = smf.ols("lwage ~ exper + educ + married + union", data=df_clean).fit()

m_out = Margins.linear_scale(
    fit_clean,
    transforms=[drop_outliers(far_below)],
    method="bootstrap",
    n_boot=500,
    rng_seed=0,
)
print(m_out.dydx("educ").summary())

flagged on the full sample: 51 rows


           Margins Result (bootstrap, level=0.95)          
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1050   0.0037     0.1050  0.000    0.0970, 0.1111

n = 4309
κ: 0.000


In [9]:
try:
    Margins.linear_scale(
        fit,
        transforms=[reimpute(imputer, incomplete=df_nan, warn_on_deterministic=False)],
        method="delta",
    )
except ValueError as exc:
    print(exc)

method='delta' is not compatible with transform stage _ReimputeStage because requires_resampling=True. Use method='bootstrap'.


In [10]:
try:
    Margins.linear_scale(
        fit,
        transforms=[drop_outliers(far_below)],
        weights=np.ones(len(df)),
        method="bootstrap",
    )
except ValueError as exc:
    print(exc)

weights= is not compatible with row-altering transform stages (_DropOutliersStage). Row-altering stages thin the data but session weights are not thinned, so weighted aggregation would misalign.
